# 기말 프로젝트 노트북. 한 계열에서 두 예측 모형을 공정하게 비교하기
*Final project notebook. Comparing two forecasting models fairly on one series*

공정한 비교에 필요한 것은 이 노트북이 고정합니다. 여러분은 계열과 모형 설정을 고르고, 결과를 점검하고 해석합니다.
*This notebook fixes what makes the comparison fair. You choose the series and the model settings, then check and interpret the results.*

| 노트북이 고정하는 것 / Fixed by the notebook | 내용 / What it is |
|---|---|
| 예측 대상 / target | 트랙 1 은 고른 계열, 트랙 2 는 소비 / the chosen series (Track 1), consumption (Track 2) |
| 예측 시점과 기간 / origins and horizon | 월별 12개월 앞, 원점 13개(3개월 간격) · 분기 4분기 앞, 원점 13개 · 트랙 2 는 4분기 앞, 원점 37개 / monthly: 12 ahead, 13 origins 3 months apart; quarterly: 4 ahead, 13 origins; Track 2: 4 ahead, 37 origins |
| 이용 정보 / information | 원점마다 그 시점까지의 자료로만 다시 맞춤(rolling origin, 7주차 · 과제 2). 미래 값은 쓰지 않음 / refit at every origin on data up to that origin only (rolling origin, Week 7 and Assignment 2) |
| 기준선 / baseline | 트랙 1 은 계절 나이브(작년 같은 달), 트랙 2 는 학습 구간 평균 / Track 1: the seasonal naive (the same month last year); Track 2: the training mean |
| 평가 구간 / evaluation window | 월별 마지막 48개월 · 분기 마지막 16분기 · 트랙 2 마지막 40분기 / the last 48 months, 16 quarters, or 40 quarters |
| 예측구간 / intervals | 80% · 95% 구간과 그 적중률 · 평균 폭 / 80% and 95% intervals, their coverage and mean width |

**쓰는 법 / How to use.** 설정은 평가 결과를 보기 **전에** 정합니다. *You fix your settings **before** you see any evaluation result.*
1. 「내 선택 1」 을 고르고 런타임 > 모두 실행. 첫 학습 구간의 그림 · 차분 확인 · 도우미 · 잔차 진단만 보입니다. 평가 구간의 실제 값은 아직 그리지 않습니다.
   *Set "Your choice 1" and Runtime > Run all. You see only the first training window's plot, the differencing check, the helper and the residual checks; the evaluation window's actual values are not drawn yet.*
2. 「내 선택 2」 에서 모형 설정과 주 지표를 정하고 「설정 확정」 을 체크한 뒤 모두 실행. 그때 백테스트가 돕니다. 전체 1분 안팎(CPU).
   *Set "Your choice 2" (model settings and the primary measure), tick "settings fixed", then Run all. Only then does the backtest run. About a minute on a CPU.*
3. 결과를 본 뒤 설정을 바꾸면 그 비교는 **탐색적 비교**입니다. 노트북이 트랙 · 자료마다 처음 확정한 설정과 그 결과(요약표와 백테스트 파일)를 자동으로 보관하고, 요약과 기록 셀에 처음 결과를 주 결과로 함께 보여 줍니다. 런타임을 초기화하면 보관 파일도 지워집니다.
   *If you change the settings after seeing the results, that comparison is **exploratory**. For each track and dataset the notebook keeps your first fixed settings and their results (summary table and backtest file) automatically, and shows the first results as the main result in the summary and the record. Resetting the runtime deletes the kept files.*

**보고서에서 / In your report.** 이긴 모형보다 「왜 그렇게 판단했나」 를 봅니다. 차이가 작거나 예상과 달라도 분석이 타당하면 만점입니다. SARIMA 와 ETS 의 AIC 는 서로 견줄 수 없습니다(12주차). 모형끼리는 아래 백테스트의 오차로 견줍니다.
*Marks go to why you judged as you did, not to which model won; a small or unexpected difference earns full marks when the analysis is sound. The AICs of SARIMA and ETS cannot be compared with each other (Week 12); compare the models by the backtest errors below.*

In [ ]:
#@title 준비와 자료 / Setup and data { display-mode: "form" }
import io, os, json, hashlib, platform, datetime, warnings, itertools, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.dates as mdates
import statsmodels
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.api import VAR
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf
from statsmodels.tools.sm_exceptions import ConvergenceWarning
NOTEBOOK = "FP-TS v6"
GREEN, GOLD, RED, GRAY, BLUE = "#406551", "#C9971C", "#B04A3A", "#9AA5A0", "#3E6A8A"

# 수업 자료와 같은 공개 자료(R 패키지 fpp3 · tsibble). 노트북 안에 넣어 내려받을 것이 없다
# Public data used in class (R packages fpp3 and tsibble), embedded so nothing needs downloading
EMP_CSV = """date,furniture,leisure,retail
2001-01,549.5,11328.0,15288.0
2001-02,541.9,11462.0,15094.4
2001-03,542.8,11645.0,15077.0
2001-04,535.5,11936.0,15046.8
2001-05,538.1,12297.0,15157.7
2001-06,538.7,12687.0,15248.1
2001-07,533.4,12757.0,15164.8
2001-08,535.5,12716.0,15189.7
2001-09,535.1,12273.0,15128.4
2001-10,539.2,11945.0,15224.1
2001-11,550.8,11702.0,15576.1
2001-12,554.2,11681.0,15713.9
2002-01,533.0,11385.0,14985.0
2002-02,526.2,11440.0,14785.9
2002-03,530.4,11622.0,14813.9
2002-04,527.5,11823.0,14874.5
2002-05,530.0,12148.0,14960.4
2002-06,533.6,12467.0,15033.4
2002-07,535.4,12536.0,15011.9
2002-08,537.2,12538.0,14981.1
2002-09,538.2,12198.0,14907.3
2002-10,547.3,12013.0,15033.6
2002-11,560.5,11837.0,15383.9
2002-12,564.5,11823.0,15577.0
2003-01,545.4,11568.0,14859.0
2003-02,543.2,11599.0,14652.3
2003-03,540.2,11769.0,14652.4
2003-04,538.4,11984.0,14725.9
2003-05,537.0,12302.0,14826.2
2003-06,543.4,12671.0,14889.5
2003-07,539.4,12776.0,14860.5
2003-08,538.9,12768.0,14898.1
2003-09,542.2,12391.0,14865.2
2003-10,552.0,12177.0,15010.3
2003-11,568.5,12026.0,15315.2
2003-12,579.2,12041.0,15505.2
2004-01,559.8,11760.0,14861.4
2004-02,553.5,11838.0,14693.8
2004-03,554.8,12077.0,14775.6
2004-04,552.6,12355.0,14862.8
2004-05,553.7,12691.0,14987.0
2004-06,557.1,13051.0,15082.0
2004-07,555.3,13128.0,15051.3
2004-08,562.0,13083.0,15054.8
2004-09,560.3,12729.0,14996.2
2004-10,570.4,12519.0,15154.5
2004-11,586.6,12348.0,15526.2
2004-12,594.3,12336.0,15707.1
2005-01,573.9,12064.0,15051.1
2005-02,567.3,12151.0,14911.9
2005-03,567.4,12365.0,14955.6
2005-04,569.5,12711.0,15068.9
2005-05,572.5,13010.0,15194.5
2005-06,570.2,13413.0,15300.1
2005-07,569.1,13514.0,15327.6
2005-08,571.7,13497.0,15348.8
2005-09,572.3,13041.0,15226.4
2005-10,581.0,12782.0,15348.9
2005-11,596.5,12629.0,15741.9
2005-12,602.0,12609.0,15938.4
2006-01,589.2,12339.0,15245.6
2006-02,580.6,12438.0,15067.2
2006-03,580.3,12674.0,15150.1
2006-04,584.1,12960.0,15171.3
2006-05,584.5,13268.0,15253.8
2006-06,584.3,13638.0,15332.6
2006-07,581.1,13790.0,15326.4
2006-08,581.7,13764.0,15337.2
2006-09,577.4,13322.0,15235.3
2006-10,588.2,13125.0,15386.0
2006-11,599.7,12994.0,15813.5
2006-12,611.6,13005.0,15982.4
2007-01,587.3,12703.0,15342.5
2007-02,577.1,12790.0,15182.0
2007-03,574.3,12987.0,15301.0
2007-04,575.2,13260.0,15327.3
2007-05,573.5,13630.0,15462.9
2007-06,567.7,14012.0,15511.5
2007-07,563.3,14092.0,15514.4
2007-08,564.8,14045.0,15538.3
2007-09,558.0,13637.0,15440.9
2007-10,567.5,13441.0,15529.0
2007-11,589.3,13278.0,15994.6
2007-12,596.6,13244.0,16162.0
2008-01,568.9,12905.0,15464.1
2008-02,552.1,12971.0,15231.2
2008-03,542.3,13156.0,15284.9
2008-04,539.5,13396.0,15247.6
2008-05,533.1,13709.0,15302.0
2008-06,531.2,14052.0,15342.8
2008-07,527.1,14129.0,15308.5
2008-08,523.1,14056.0,15270.9
2008-09,515.7,13563.0,15099.4
2008-10,516.2,13300.0,15138.1
2008-11,512.5,13040.0,15351.7
2008-12,510.9,12959.0,15424.4
2009-01,479.9,12607.0,14688.6
2009-02,459.0,12616.0,14439.0
2009-03,451.9,12748.0,14410.6
2009-04,446.9,12977.0,14399.6
2009-05,440.6,13337.0,14495.4
2009-06,439.0,13656.0,14543.5
2009-07,439.0,13746.0,14489.3
2009-08,436.4,13675.0,14495.3
2009-09,435.6,13245.0,14367.5
2009-10,443.9,12956.0,14413.0
2009-11,455.3,12735.0,14729.7
2009-12,462.9,12631.0,14863.7
2010-01,441.8,12315.0,14290.5
2010-02,432.2,12359.0,14123.0
2010-03,432.4,12578.0,14209.3
2010-04,430.6,12915.0,14269.1
2010-05,430.9,13243.0,14380.1
2010-06,433.3,13616.0,14441.8
2010-07,430.9,13720.0,14445.1
2010-08,429.1,13712.0,14463.7
2010-09,432.0,13351.0,14361.0
2010-10,446.7,13085.0,14510.5
2010-11,453.9,12872.0,14849.9
2010-12,460.5,12818.0,15008.4
2011-01,444.1,12477.0,14448.6
2011-02,432.7,12614.0,14282.5
2011-03,433.3,12881.0,14349.3
2011-04,432.7,13233.0,14489.0
2011-05,432.0,13535.0,14588.7
2011-06,431.7,13947.0,14682.3
2011-07,433.6,14032.0,14718.0
2011-08,431.4,14023.0,14717.5
2011-09,432.5,13629.0,14617.2
2011-10,447.3,13418.0,14751.7
2011-11,456.1,13249.0,15142.3
2011-12,459.5,13195.0,15296.7
2012-01,444.3,12904.0,14734.5
2012-02,433.9,13038.0,14520.3
2012-03,432.2,13334.0,14580.3
2012-04,430.5,13626.0,14670.4
2012-05,432.1,13959.0,14775.5
2012-06,430.7,14329.0,14815.9
2012-07,431.2,14411.0,14807.7
2012-08,434.6,14434.0,14815.2
2012-09,432.5,14055.0,14738.9
2012-10,444.8,13838.0,14877.7
2012-11,460.2,13646.0,15363.0
2012-12,465.3,13646.0,15462.5
2013-01,447.3,13324.0,14858.2
2013-02,440.1,13470.0,14672.3
2013-03,436.4,13740.0,14711.7
2013-04,439.2,14074.0,14808.6
2013-05,439.3,14459.0,14932.3
2013-06,438.7,14881.0,15045.2
2013-07,439.8,14978.0,15097.3
2013-08,437.3,14961.0,15131.7
2013-09,437.4,14518.0,15052.6
2013-10,451.4,14346.0,15212.3
2013-11,466.9,14174.0,15655.3
2013-12,475.5,14124.0,15839.4
2014-01,452.7,13815.0,15170.7
2014-02,445.4,13908.0,14951.8
2014-03,444.0,14181.0,15015.4
2014-04,444.8,14524.0,15137.0
2014-05,444.8,14935.0,15238.9
2014-06,446.1,15333.0,15354.6
2014-07,447.9,15407.0,15389.2
2014-08,455.2,15365.0,15380.4
2014-09,456.3,14948.0,15298.8
2014-10,468.4,14787.0,15481.6
2014-11,479.3,14587.0,15894.3
2014-12,485.4,14563.0,16048.1
2015-01,465.8,14234.0,15421.1
2015-02,463.1,14374.0,15244.9
2015-03,460.9,14599.0,15311.7
2015-04,460.3,14947.0,15401.8
2015-05,462.4,15365.0,15530.3
2015-06,464.2,15768.0,15620.5
2015-07,464.1,15878.0,15647.0
2015-08,464.3,15845.0,15626.3
2015-09,463.0,15441.0,15513.4
2015-10,469.0,15293.0,15696.8
2015-11,480.0,15103.0,16095.8
2015-12,484.5,15077.0,16221.5
2016-01,469.5,14747.0,15625.7
2016-02,465.0,14889.0,15486.6
2016-03,464.7,15143.0,15576.8
2016-04,462.8,15480.0,15648.7
2016-05,465.6,15845.0,15745.7
2016-06,468.3,16320.0,15851.8
2016-07,469.3,16421.0,15874.4
2016-08,471.7,16383.0,15864.6
2016-09,468.3,15930.0,15750.3
2016-10,474.8,15721.0,15899.5
2016-11,483.3,15550.0,16260.2
2016-12,488.0,15489.0,16394.3
2017-01,482.8,15124.0,15854.4
2017-02,476.1,15283.0,15627.9
2017-03,471.2,15524.0,15635.0
2017-04,469.3,15887.0,15686.6
2017-05,468.6,16285.0,15759.5
2017-06,467.9,16758.0,15843.0
2017-07,468.1,16886.0,15841.1
2017-08,469.1,16808.0,15810.2
2017-09,468.4,16226.0,15679.3
2017-10,477.9,16098.0,15819.9
2017-11,494.7,15875.0,16285.8
2017-12,500.8,15859.0,16305.9
2018-01,484.8,15437.0,15718.6
2018-02,476.8,15588.0,15577.0
2018-03,474.9,15818.0,15610.8
2018-04,473.7,16132.0,15681.4
2018-05,474.0,16552.0,15797.2
2018-06,474.5,17051.0,15844.9
2018-07,474.4,17164.0,15854.5
2018-08,474.7,17096.0,15834.9
2018-09,474.9,16546.0,15680.6
2018-10,486.2,16402.0,15796.5
2018-11,500.2,16182.0,16291.3
2018-12,507.3,16213.0,16309.2
2019-01,494.8,15880.0,15753.5
2019-02,483.7,15997.0,15567.4
2019-03,484.9,16260.0,15576.6
2019-04,482.1,16575.0,15624.9
2019-05,481.5,16961.0,15691.6
2019-06,481.4,17403.0,15775.5
2019-07,479.1,17463.0,15785.9
2019-08,478.5,17380.0,15749.5
2019-09,476.1,16872.0,15611.3"""
HOL_CSV = """date,trips
1998-01,11.8060
1998-04,9.2757
1998-07,8.6425
1998-10,9.2995
1999-01,11.1720
1999-04,9.6076
1999-07,8.9139
1999-10,9.0257
2000-01,11.0709
2000-04,9.1963
2000-07,9.3475
2000-10,8.9840
2001-01,10.6723
2001-04,9.5335
2001-07,8.8072
2001-10,8.9769
2002-01,10.8426
2002-04,9.2327
2002-07,8.8362
2002-10,9.2475
2003-01,10.5016
2003-04,9.0935
2003-07,8.8405
2003-10,8.8413
2004-01,10.7015
2004-04,9.2113
2004-07,8.9244
2004-10,8.3786
2005-01,10.9290
2005-04,8.0663
2005-07,8.1005
2005-10,7.7370
2006-01,11.2192
2006-04,9.2740
2006-07,7.7962
2006-10,8.9284
2007-01,10.9103
2007-04,9.3564
2007-07,8.6175
2007-10,8.7164
2008-01,11.9515
2008-04,8.4995
2008-07,8.4822
2008-10,8.3911
2009-01,10.1320
2009-04,8.4004
2009-07,8.6903
2009-10,8.2623
2010-01,10.1570
2010-04,8.7339
2010-07,8.6065
2010-10,8.5047
2011-01,9.8464
2011-04,9.0092
2011-07,8.4495
2011-10,8.4886
2012-01,10.2679
2012-04,8.6029
2012-07,8.3504
2012-10,9.2375
2013-01,10.7867
2013-04,9.0333
2013-07,8.7810
2013-10,8.8768
2014-01,11.3553
2014-04,10.1989
2014-07,8.9680
2014-10,9.4143
2015-01,11.6309
2015-04,9.7070
2015-07,9.5460
2015-10,10.0462
2016-01,12.5932
2016-04,9.9802
2016-07,9.9396
2016-10,10.0850
2017-01,12.4064
2017-04,10.4712
2017-07,10.4992
2017-10,11.2108"""
CHG_CSV = """date,consumption,income
1970-01,0.618566,1.044801
1970-04,0.451984,1.225647
1970-07,0.872872,1.585154
1970-10,-0.271848,-0.239545
1971-01,1.901345,1.975925
1971-04,0.914877,1.445909
1971-07,0.794110,0.521149
1971-10,1.645633,1.159176
1972-01,1.311190,0.456857
1972-04,1.885778,1.033389
1972-07,1.529334,1.926295
1972-10,2.319558,3.929949
1973-01,1.809282,0.755643
1973-04,-0.045091,0.840392
1973-07,0.351353,0.451027
1973-10,-0.298693,1.088161
1974-01,-0.898699,-1.674191
1974-04,0.359406,-0.934440
1974-07,0.428641,0.090326
1974-10,-1.472503,-0.133675
1975-01,0.835247,-0.150542
1975-04,1.631487,4.521869
1975-07,1.411240,-1.464342
1975-10,1.069592,0.778694
1976-01,1.979659,1.200323
1976-04,0.915845,0.566353
1976-07,1.047352,0.777196
1976-10,1.289140,0.632455
1977-01,1.171799,0.224476
1977-04,0.540781,0.935510
1977-07,0.941116,1.380019
1977-10,1.492345,1.893421
1978-01,0.560223,0.781021
1978-04,2.115329,1.052862
1978-07,0.419027,0.774758
1978-10,0.803703,0.655688
1979-01,0.516597,1.202750
1979-04,-0.064958,-0.908216
1979-07,0.968393,0.453218
1979-10,0.248513,0.176009
1980-01,-0.143054,0.345513
1980-04,-2.277836,-0.882967
1980-07,1.090975,1.116963
1980-10,1.342028,1.367396
1981-01,0.334439,-0.140020
1981-04,0.007281,0.116626
1981-07,0.461512,2.133511
1981-10,-0.686367,0.109843
1982-01,0.733889,0.241051
1982-04,0.304135,0.601491
1982-07,0.668142,0.478676
1982-10,1.736507,0.313585
1983-01,0.983849,0.939211
1983-04,2.015813,0.690659
1983-07,1.777314,1.489413
1983-10,1.581000,2.202848
1984-01,0.826451,1.977824
1984-04,1.391254,1.647347
1984-07,0.738898,1.311542
1984-10,1.287102,0.813019
1985-01,1.684776,-0.261339
1985-04,0.913850,1.928891
1985-07,1.881522,-0.313839
1985-10,0.216097,1.052373
1986-01,0.870018,1.306420
1986-04,1.083790,1.270835
1986-07,1.753563,0.635496
1986-10,0.611476,0.152240
1987-01,0.099765,0.917143
1987-04,1.355464,-0.940301
1987-07,1.134844,1.774146
1987-10,0.211331,1.339490
1988-01,1.751078,1.456190
1988-04,0.735545,1.024320
1988-07,0.884646,0.961912
1988-10,1.130580,0.939045
1989-01,0.462871,1.132062
1989-04,0.465700,-0.295235
1989-07,0.963449,0.662055
1989-10,0.433673,0.842871
1990-01,0.838072,0.765049
1990-04,0.304152,0.693536
1990-07,0.395373,-0.035389
1990-10,-0.763666,-0.856437
1991-01,-0.376738,0.249602
1991-04,0.825347,0.728949
1991-07,0.497353,0.407428
1991-10,-0.050052,0.827904
1992-01,1.855027,1.992517
1992-04,0.676546,0.986750
1992-07,1.066222,0.462840
1992-10,1.184841,0.454413
1993-01,0.382689,0.373438
1993-04,0.905498,0.308897
1993-07,1.106802,0.087083
1993-10,0.874024,0.692553
1994-01,1.137305,0.658186
1994-04,0.775059,1.002084
1994-07,0.759622,0.581464
1994-10,1.075324,1.424746
1995-01,0.248575,0.851213
1995-04,0.872850,0.283147
1995-07,0.902152,0.912093
1995-10,0.695146,0.582727
1996-01,0.918062,0.942444
1996-04,1.070039,0.912727
1996-07,0.603594,0.806478
1996-10,0.793361,0.531211
1997-01,1.048573,0.947932
1997-04,0.449184,0.882317
1997-07,1.689403,1.169845
1997-10,1.188130,1.462042
1998-01,1.006450,2.083377
1998-04,1.745611,1.365644
1998-07,1.295597,1.002353
1998-10,1.434358,0.788512
1999-01,0.965597,0.899890
1999-04,1.447472,0.225012
1999-07,1.122360,0.679935
1999-10,1.454582,1.490762
2000-01,1.524570,1.896433
2000-04,0.961212,1.099804
2000-07,0.977169,1.154542
2000-10,0.856410,0.356800
2001-01,0.399500,0.902078
2001-04,0.236146,-0.174790
2001-07,0.342130,2.281184
2001-10,1.484246,-1.271797
2002-01,0.305592,2.229188
2002-04,0.512136,0.674730
2002-07,0.721913,-0.087220
2002-10,0.573051,0.592006
2003-01,0.469223,0.234893
2003-04,1.116773,1.211482
2003-07,1.456168,1.675616
2003-10,0.675769,0.263150
2004-01,0.885842,0.459518
2004-04,0.620249,1.139362
2004-07,1.099777,0.650787
2004-10,1.081821,1.255122
2005-01,0.581777,-1.179351
2005-04,1.128161,0.945498
2005-07,0.965993,0.295213
2005-10,0.296515,1.272756
2006-01,1.108947,1.926703
2006-04,0.505326,0.244248
2006-07,0.605281,0.258378
2006-10,0.963100,1.311031
2007-01,0.558011,0.848367
2007-04,0.185169,0.255744
2007-07,0.562009,0.088781
2007-10,0.284407,0.067964
2008-01,-0.266123,0.277378
2008-04,0.162019,1.813508
2008-07,-0.751601,-2.123959
2008-10,-0.935007,0.867342
2009-01,-0.226118,-0.429951
2009-04,-0.405844,1.069271
2009-07,0.687520,-1.137177
2009-10,-0.148425,-0.024645
2010-01,0.493232,0.560634
2010-04,0.792222,1.634157
2010-07,0.696906,0.709404
2010-10,0.659198,0.559127
2011-01,0.424447,1.013855
2011-04,0.221177,-0.217924
2011-07,0.390804,0.435638
2011-10,0.182577,0.310519
2012-01,0.799759,1.854619
2012-04,0.148268,0.905012
2012-07,0.163040,-0.718402
2012-10,0.447192,2.732609
2013-01,0.517198,-4.084420
2013-04,0.071945,0.742249
2013-07,0.405903,0.431404
2013-10,0.857976,0.394307
2014-01,0.393157,1.383321
2014-04,1.088954,1.352928
2014-07,1.070283,1.160578
2014-10,1.205427,1.313011
2015-01,0.827268,1.129293
2015-04,0.792813,0.749587
2015-07,0.759427,0.732371
2015-10,0.446824,0.316750
2016-01,0.778254,0.666137
2016-04,0.715591,-0.106588
2016-07,0.634916,0.447270
2016-10,0.618545,0.598038
2017-01,0.593337,1.192032
2017-04,0.593153,0.664810
2017-07,0.584738,0.579716
2017-10,1.133146,0.907210
2018-01,0.416963,1.665726
2018-04,0.983112,0.661825
2018-07,0.853181,0.806271
2018-10,0.356512,0.695142
2019-01,0.282885,1.100753
2019-04,1.113517,0.593399"""
_emp = pd.read_csv(io.StringIO(EMP_CSV), parse_dates=["date"]).set_index("date")
_hol = pd.read_csv(io.StringIO(HOL_CSV), parse_dates=["date"]).set_index("date")
_chg = pd.read_csv(io.StringIO(CHG_CSV), parse_dates=["date"]).set_index("date")
BUILT_IN = {
    "Furniture and home furnishings stores employment (Assignment 2)": (_emp["furniture"], "monthly", "thousands of persons"),
    "Leisure and hospitality employment (Week 7)": (_emp["leisure"], "monthly", "thousands of persons"),
    "Retail trade employment (Week 12)": (_emp["retail"], "monthly", "thousands of persons"),
    "Australian domestic holiday trips, quarterly (Week 12)": (_hol["trips"], "quarterly", "millions of trips")}       # 원자료(천 건)를 1,000 으로 나눈 값 / the source counts thousands; divided by 1,000
DESIGN = {"monthly": {"m": 12, "h": 12, "step": 3, "n_origins": 13, "lb_lag": 24},
          "quarterly": {"m": 4, "h": 4, "step": 1, "n_origins": 13, "lb_lag": 8},
          "track2": {"m": 1, "h": 4, "step": 1, "n_origins": 37, "lb_lag": 8}}

def make_origins(n, d):
    """마지막 원점은 n − h, 그 앞으로 step 간격 / the last origin is n − h, then every `step` before it."""
    return [n - d["h"] - d["step"] * k for k in range(d["n_origins"])][::-1]

def quiet(f, *a, **k):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore"); return f(*a, **k)

def fit_checked(build, first, retry):
    """수렴 경고를 숨기지 않고 확인한다. 실패하면 반복을 늘려 한 번 더. 결과에 converged_ok 를 붙인다.
    Check convergence instead of hiding warnings; on failure retry once with more iterations; tag the result with converged_ok."""
    res = None
    for kw in (first, retry):
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            res = build().fit(**kw)
        rv = res.mle_retvals if isinstance(getattr(res, "mle_retvals", None), dict) else {}
        if not any(issubclass(x.category, ConvergenceWarning) for x in w) and rv.get("converged", True) is not False:
            res.converged_ok = True; return res
    res.converged_ok = False; return res

def fit_sarima(y, order, sorder):
    return fit_checked(lambda: SARIMAX(np.asarray(y, float), order=order, seasonal_order=sorder,
                                       trend="c" if order[1] + sorder[1] == 0 else "n"), {"disp": False}, {"disp": False, "maxiter": 500})
def fc_sarima(res, h):
    f = res.get_forecast(h); out = {"mean": np.asarray(f.predicted_mean)}
    for a, tag in ((0.20, "80"), (0.05, "95")):
        ci = np.asarray(f.conf_int(alpha=a)); out["lo" + tag], out["hi" + tag] = ci[:, 0], ci[:, 1]
    return out
# ETS 후보 열다섯(12주차 덱 「Every Method Has an ETS Name」 의 표기, 12주차 실습과 같은 후보). 오차 A · M, 추세 N · A · Ad,
# 계절 N · A · M 의 열여덟 조합 가운데 ETS(A,N,M) · ETS(A,A,M) · ETS(A,Ad,M) 은 수치가 불안정해 교재(FPP-py §8.6)처럼 고르지 않는다.
# The fifteen ETS candidates, in the notation of Week 12 and the same as the Week 12 lab. Of the eighteen combinations,
# ETS(A,N,M), ETS(A,A,M) and ETS(A,Ad,M) are numerically unstable and, as in the textbook (FPP-py §8.6), are not considered.
ETS_CANDIDATES = [f"ETS({e},{t},{s})" for e in ("A", "M") for s in ("N", "A", "M") for t in ("N", "A", "Ad")
                  if not (e == "A" and s == "M")]
def ets_args(name):
    """수업 표기 → statsmodels ETSModel 인자 / course notation to ETSModel arguments. 예 / e.g. ETS(M,Ad,A) → error mul, trend add damped, seasonal add."""
    e, t, s = name[4:-1].split(",")
    return {"error": {"A": "add", "M": "mul"}[e], "trend": None if t == "N" else "add", "damped_trend": t == "Ad",
            "seasonal": {"N": None, "A": "add", "M": "mul"}[s]}
ETS_SECOND_START = {"smoothing_level": 0.5, "smoothing_trend": 0.05, "smoothing_seasonal": 0.1}   # 두 번째 출발점: 빠른 평활 / the second start: fast smoothing
ETS_OPTIMIZER = {"m": 50, "factr": 1e4}   # L-BFGS 가 기억하는 걸음 10 -> 50, 멈춤 기준 1e7 -> 1e4(더 엄격) / remember 50 steps, not 10, and stop later
def fit_ets(y, name, m):
    """자료를 평균 크기로 나눠(평균 1 쯤) 맞춘 뒤, 같은 모수를 원래 단위로 옮긴다. 그래서 단위(천 명 · 백만 명)만 다른 자료에서 같은 모형 · 같은 예측이 나온다.
    최적화는 덜 간 채 「수렴」 으로 멈출 수 있고 어디서 멈추는지가 컴퓨터마다 다르다. 그래서 출발점 둘에서 맞추고 멈춘 자리에서 다시 출발해 본 뒤,
    수렴한 시도 가운데 훈련 로그우도가 가장 높은 적합을 쓴다(12주차 실습과 같은 규칙). 예측할 구간의 값은 쓰지 않는다.
    Fit the data divided by their average size (about 1), then move the same parameters back to the original units,
    so data that differ only in units (thousands or millions) give the same model and the same forecasts.
    The optimizer can report convergence before it gets there, and where it stops differs between computers. So it starts from two
    points, restarts from where it stopped, and keeps the converged try with the highest training log-likelihood (the Week 12 lab's rule)."""
    a = ets_args(name); y = np.asarray(y, float); size = float(np.mean(np.abs(y)))     # 자료의 평균 크기 / the data's average size
    idx = pd.date_range("2000-01-01", periods=len(y), freq="MS" if m == 12 else "QS")
    kw = dict(seasonal_periods=m if a["seasonal"] else None, **a)
    z = pd.Series(np.round(y / size, 10), index=idx)              # 소수 열째 자리까지: 단위를 바꿀 때 남는 아주 작은 계산 오차를 지운다 / to 10 decimals: removes the tiny rounding a change of units leaves
    def one_try(start):                                           # 수렴 확인, 실패하면 반복을 늘려 한 번 더 / check convergence, retry once with more iterations
        return fit_checked(lambda: ETSModel(z, **kw), {"start_params": start, "disp": False, **ETS_OPTIMIZER},
                           {"start_params": start, "disp": False, "maxiter": 5000, **ETS_OPTIMIZER})
    tries = []
    for change in (None, ETS_SECOND_START):                       # 기본 출발점, 그리고 빠른 평활 출발점 / the default start, then the fast-smoothing start
        start = None
        if change is not None:
            m0 = ETSModel(z, **kw); start = np.asarray(m0.start_params, float).copy()
            for k, v in change.items():                           # 모형에 있는 평활 계수만 바꾼다 / change only the smoothing weights this model has
                if k in m0.param_names: start[m0.param_names.index(k)] = v
        f = one_try(start); tries.append(f)
        for _ in range(2):                                        # 멈춘 자리에서 다시 출발, 우도가 더 오르지 않으면 그만 / restart where it stopped; quit when the likelihood stops rising
            again = one_try(np.asarray(f.params, float)); tries.append(again)
            if not again.llf > f.llf + 1e-8: break
            f = again
    unit = max(tries, key=lambda r: (r.converged_ok, r.llf if np.isfinite(r.llf) else -np.inf))   # 수렴한 것 먼저, 그 가운데 훈련 로그우도 최대 / converged first, then the highest training log-likelihood
    p = np.asarray(unit.params, float).copy()                     # 평활 계수 · 감쇠 · 곱셈 계절은 단위가 없어 그대로 / unit-free parts stay
    for i, nm in enumerate(unit.model.param_names):               # 수준 · 기울기 · 덧셈 계절의 시작값만 원래 단위로 / starting level, slope and added season back to the units
        if nm in ("initial_level", "initial_trend") or (nm.startswith("initial_seasonal") and a["seasonal"] == "add"):
            p[i] *= size
    res = quiet(ETSModel(pd.Series(y, index=idx), **kw).smooth, p)   # 다시 맞추지 않고 같은 모수로 / the same parameters, no refit
    res.converged_ok, res.mle_retvals = unit.converged_ok, getattr(unit, "mle_retvals", {})
    return res
def sarima_name(order, sorder):
    """과제 2 와 같은 이름 / the Assignment 2 name, e.g. ARIMA(1,1,1)(0,1,1)12."""
    return f"ARIMA({order[0]},{order[1]},{order[2]})({sorder[0]},{sorder[1]},{sorder[2]}){sorder[3]}"
def bars(ax, r, band, title, xlabel):
    """ACF 막대. 띠 ±2/√n(6주차) 밖은 빨강, 띠는 금색 점선. 7주차 · 과제 2 와 같은 모양.
    ACF bars: red outside the band ±2/√n (Week 6), the band as gold dashes. The Week 7 and Assignment 2 look."""
    lags = np.arange(1, len(r) + 1)
    ax.bar(lags, r, width=0.5, color=[RED if abs(v) > band else GREEN for v in r])
    ax.axhline(band, color=GOLD, ls="--"); ax.axhline(-band, color=GOLD, ls="--"); ax.axhline(0, color=GRAY, lw=0.6)
    ax.set_title(title, fontsize=9); ax.set_xlabel(xlabel)
def fc_ets(res, h, n):
    out = {}
    for a, tag in ((0.20, "80"), (0.05, "95")):
        p = None
        for kw in ({"rng": np.random.default_rng(0)}, {"random_state": 0}):     # 판마다 인자 이름이 다르다 / the argument name differs by version
            try:
                p = quiet(res.get_prediction, start=n, end=n + h - 1, simulate_repetitions=2000, **kw); break
            except TypeError:
                continue
        sf = p.summary_frame(alpha=a)
        out["mean"] = sf["mean"].to_numpy(); out["lo" + tag], out["hi" + tag] = sf["pi_lower"].to_numpy(), sf["pi_upper"].to_numpy()
    return out
def fc_snaive(y, h, m):
    base = np.asarray(y, float)[-m:]; return {"mean": np.array([base[i % m] for i in range(h)])}
def fit_ar(c, p):
    return fit_checked(lambda: SARIMAX(np.asarray(c, float), order=(p, 0, 0), trend="c"), {"disp": False}, {"disp": False, "maxiter": 500})

def check_regular(Yx, freq, label):
    """본인 자료 검사: 숫자, 빈 값, 중복 기간, 빠진 기간 / own data: numbers, missing values, duplicated or skipped periods."""
    per = Yx.index.to_period("M" if freq == "monthly" else "Q")
    if Yx.isna().to_numpy().any():
        raise ValueError(f"{label}: 빈 칸이나 숫자가 아닌 칸이 있습니다(첫 위치 {Yx.index[Yx.isna().to_numpy().any(axis=1) if Yx.ndim > 1 else Yx.isna().to_numpy()][0].date()}) / "
                         f"missing or non-numeric values (first at {Yx.index[Yx.isna().to_numpy().any(axis=1) if Yx.ndim > 1 else Yx.isna().to_numpy()][0].date()})")
    if per.has_duplicates:
        raise ValueError(f"{label}: 같은 기간이 두 번 있습니다 / a period appears twice: {per[per.duplicated()][0]}")
    gaps = np.diff(per.asi8)
    if (gaps != 1).any():
        raise ValueError(f"{label}: {per[1:][gaps != 1][0]} 앞에 빠진 기간이 있습니다. 「12개 앞」 이 「12개월 앞」 이 되려면 간격이 고르게 이어져야 합니다 / "
                         f"a period is missing before {per[1:][gaps != 1][0]}; the series must be regular so that 12 steps ahead means 12 {'months' if freq == 'monthly' else 'quarters'} ahead")
def fit_var(Y, p):
    return VAR(np.asarray(Y, float)).fit(p)
def fc_var(res, Y, h):
    out = {}
    for a, tag in ((0.20, "80"), (0.05, "95")):
        mean, lo, hi = res.forecast_interval(np.asarray(Y, float)[-res.k_ar:], steps=h, alpha=a)
        out["mean"], out["lo" + tag], out["hi" + tag] = mean[:, 0], lo[:, 0], hi[:, 0]
    return out
def mase_scale(y_train, m):
    y = np.asarray(y_train, float); return np.mean(np.abs(y[m:] - y[:-m]))

def summarize(bt):
    e = bt["actual"] - bt["forecast"]
    bt = bt.assign(sq=e ** 2, ab=e.abs(), sc=e.abs() / bt["scale"])
    g = bt.groupby("model", sort=False)
    out = pd.DataFrame({"RMSE": np.sqrt(g["sq"].mean()), "MAE": g["ab"].mean(), "MASE": g["sc"].mean()})
    for tag in ("80", "95"):
        has = bt["lo" + tag].notna()
        inside = ((bt["actual"] >= bt["lo" + tag]) & (bt["actual"] <= bt["hi" + tag])).astype(float).where(has)
        out[f"coverage{tag}_%"] = 100 * inside.groupby(bt["model"], sort=False).mean()
        out[f"width{tag}"] = (bt["hi" + tag] - bt["lo" + tag]).groupby(bt["model"], sort=False).mean()
    return out
print("준비 완료 / Setup ready | statsmodels", statsmodels.__version__)

## 내 선택 1. 트랙과 자료 / Your choice 1. Track and data
- **트랙 1 (SARIMA 대 ETS)**: 제공 계열 넷 가운데 하나. 월별 세 계열은 2001년 1월 ~ 2019년 9월(225개월), 휴가 여행은 1998 ~ 2017년(80분기, 백만 건).
  *Track 1: one of the four provided series. The three monthly series run January 2001 to September 2019 (225 months); holiday trips 1998 to 2017 (80 quarters, millions).*
- **트랙 2 (AR 대 VAR)**: 미국 분기별 소비 · 소득 변화율(1970 ~ 2019년, 198분기, 10주차 자료). 계열 칸은 쓰지 않습니다.
  *Track 2: US quarterly changes in consumption and income (1970 to 2019, 198 quarters, the Week 10 data). The series box is not used.*
- **본인 자료**: 제안서 전에 협의한 경우만. CSV 를 Colab 왼쪽 파일 칸에 올리고 경로를 적으세요. 트랙 1 은 열 두 개(날짜, 값), 트랙 2 는 열 세 개(날짜, 예측할 계열, 함께 쓸 계열). 기간이 빠지거나 겹치면 멈추고 알려 줍니다.
  *Your own data: only if cleared with me. Upload a CSV in Colab's file pane and enter its path. Track 1 needs two columns (date, value); Track 2 needs three (date, the series to forecast, the helper series). The notebook stops with a message if a period is missing or repeated.*

In [ ]:
#@title 내 선택 1 / Your choice 1 { display-mode: "form" }
TRACK = "Track 1: SARIMA versus ETS"  #@param ["Track 1: SARIMA versus ETS", "Track 2: AR versus VAR"]
SERIES_NAME = "Furniture and home furnishings stores employment (Assignment 2)"  #@param ["Furniture and home furnishings stores employment (Assignment 2)", "Leisure and hospitality employment (Week 7)", "Retail trade employment (Week 12)", "Australian domestic holiday trips, quarterly (Week 12)"]
OWN_CSV_PATH = ""  #@param {type:"string"}
OWN_FREQUENCY = "monthly"  #@param ["monthly", "quarterly"]

T1 = TRACK.startswith("Track 1")
if OWN_CSV_PATH.strip():
    if not os.path.exists(OWN_CSV_PATH.strip()):
        raise FileNotFoundError(f"파일이 없습니다 / file not found: {OWN_CSV_PATH!r}. Colab 왼쪽 파일 칸에 올렸는지, 경로가 맞는지 보세요 / check the upload and the path.")
    _own = pd.read_csv(OWN_CSV_PATH.strip())
    need_cols = 2 if T1 else 3
    if _own.shape[1] < need_cols:
        raise ValueError(f"열이 모자랍니다 / need {need_cols} columns: date + {need_cols - 1}")
    _c0 = _own.columns[0]; _own[_c0] = pd.to_datetime(_own[_c0], errors="coerce")
    if _own[_c0].isna().any():
        raise ValueError("날짜로 읽히지 않는 칸이 있습니다 / some dates could not be read")
    _own = _own.set_index(_c0).sort_index()
    _vals = _own.iloc[:, :need_cols - 1].apply(pd.to_numeric, errors="coerce")
    FREQ = OWN_FREQUENCY
    check_regular(_vals, FREQ, os.path.basename(OWN_CSV_PATH.strip()))
    Y = _vals.iloc[:, 0] if T1 else _vals
    DATA_LABEL, UNITS = f"own data: {os.path.basename(OWN_CSV_PATH.strip())}", "own units"
    DATA_SHA = hashlib.sha256(open(OWN_CSV_PATH.strip(), "rb").read()).hexdigest()[:16]
else:
    if T1:
        Y, FREQ, UNITS = BUILT_IN[SERIES_NAME]; DATA_LABEL = SERIES_NAME
    else:
        Y, FREQ, UNITS = _chg[["consumption", "income"]], "quarterly", "% change"; DATA_LABEL = "US consumption and income (Week 10)"
    DATA_SHA = hashlib.sha256(Y.to_csv().encode()).hexdigest()[:16]
D = DESIGN[FREQ] if T1 else DESIGN["track2"]
N = len(Y); ORIGINS = make_origins(N, D); M = D["m"]
first_train = ORIGINS[0]
min_train = 3 * M + 10 if T1 else 30
if first_train < min_train:
    raise ValueError(f"자료가 짧습니다 / series too short: the first training window has {first_train} points, need at least {min_train}.")
dates = Y.index
unit = "month" if FREQ == "monthly" else "quarter"
print(f"자료 / data      : {DATA_LABEL} ({UNITS}), {N} points, {dates[0].date()} ~ {dates[-1].date()}")
print(f"설계 / design    : {D['h']} {unit}s ahead, {len(ORIGINS)} origins every {D['step']} {unit}(s)")
print(f"첫 학습 구간 / first training window : {dates[0].date()} ~ {dates[first_train - 1].date()} ({first_train} points)")
print(f"평가 구간 / evaluation window       : {dates[first_train].date()} ~ {dates[-1].date()} ({N - first_train} points). 값은 설정을 확정한 뒤에 봅니다 / its values are shown after you fix the settings")

fig, ax = plt.subplots(figsize=(9, 2.8))
target = Y if T1 else Y.iloc[:, 0]
ax.plot(dates[:first_train], target.iloc[:first_train], color=GREEN, lw=1.2, label="series to forecast" if T1 else "consumption (target)")
if not T1: ax.plot(dates[:first_train], Y.iloc[:first_train, 1], color=GRAY, lw=0.9, alpha=0.8, label="income (helper)")
ax.axvline(dates[first_train - 1], color=GOLD, lw=1.5, ls="--", label="evaluation window starts after this line")
ax.set(title=DATA_LABEL + ": first training window only", ylabel=UNITS); ax.legend(fontsize=8, frameon=False); plt.tight_layout(); plt.show()

## 내 선택 2. 모형 설정 / Your choice 2. Model settings
**트랙 1.** 먼저 차분 d · 계절 차분 D 를 7주차와 과제 2 처럼 정하세요. 바로 아래 「차분 확인」 셀이 첫 학습 구간의 원계열과, 고른 차분을 한 계열의 그림 · ACF 를 보여 줍니다(과제 2 Q1 과 같은 그림). 보고 다시 정해도 됩니다. 설정을 확정하기 전에는 몇 번이든 바꿔도 됩니다. 그 차분 위에서 아래 도우미가 SARIMA 차수 p · q · P · Q 의 AIC 와 ETS 후보 열다섯의 AICc 를 **첫 학습 구간에서만** 계산해 제안합니다. 제안값을 쓰거나(체크) 직접 고르세요(체크 해제 + 값). 어느 쪽이든 보고서에 이유를 쓰세요.
*Track 1. First set the differences d and D as in Week 7 and Assignment 2. The "Differencing check" cell just below shows the first training window's original series and the series after your differences, each with its plot and ACF (the plots of Assignment 2, Q1). You may change d and D after looking, as often as you like before you fix the settings. On those differences, the helper below computes the AIC of SARIMA orders p, q, P, Q and the AICc of the fifteen ETS candidates **on the first training window only**, and suggests one of each. Use the suggestion (ticked) or choose your own (unticked plus values); either way, give your reason in the report.*

**ETS 의 이름과 코드 / ETS names and the code.** 선택 칸과 출력은 12주차 덱 「Every Method Has an ETS Name」 의 이름 ETS(오차, 추세, 계절)을 씁니다. 계산은 statsmodels 의 `ETSModel` 이고, 이름의 글자는 아래처럼 인자로 바뀝니다.
*The menu and the output use the Week 12 names ETS(Error, Trend, Season). The computation is statsmodels' `ETSModel`, and each letter becomes an argument as below.*

| 이름의 글자 / letter in the name | 뜻 / meaning | `ETSModel` 인자 / argument |
|---|---|---|
| 오차 A · M / error A or M | 더하는 오차 · 곱하는 오차 / added or multiplied errors | `error="add"` · `error="mul"` |
| 추세 N · A · Ad / trend N, A or Ad | 없음 · 덧셈 · 감쇠한 덧셈 / none, added, damped | `trend=None` · `trend="add"` · `trend="add", damped_trend=True` |
| 계절 N · A · M / season N, A or M | 없음 · 덧셈 · 곱셈 / none, added, multiplied | `seasonal=None` · `seasonal="add"` · `seasonal="mul"` |

예: ETS(M,Ad,A) 는 `error="mul", trend="add", damped_trend=True, seasonal="add"` 입니다. 열여덟 조합 가운데 덧셈 오차에 곱셈 계절을 붙인 셋, ETS(A,N,M) · ETS(A,A,M) · ETS(A,Ad,M) 은 계산 중에 0 에 가까운 값으로 나눌 수 있어 수치가 불안정합니다. 그래서 교재(FPP-py §8.6)와 12주차 실습처럼 후보는 열다섯입니다. 같은 AICc 규칙이라도 계산하는 프로그램(교재의 statsforecast 등)이 다르면 고르는 모형이 다를 수 있습니다. 이 노트북은 자료를 평균 크기로 나눠 맞춘 뒤 원래 단위로 되돌리므로, 단위(천 명 · 백만 명)만 다른 자료에서는 같은 모형을 고릅니다. 최적화가 덜 간 채 멈추지 않도록 후보마다 출발점 둘에서 맞추고 멈춘 자리에서 다시 출발해, 수렴한 것 가운데 훈련 로그우도가 가장 높은 적합을 씁니다(12주차 실습과 같은 도구).
*Example: ETS(M,Ad,A) is `error="mul", trend="add", damped_trend=True, seasonal="add"`. Of the eighteen combinations, the three that put a multiplicative season on additive errors, ETS(A,N,M), ETS(A,A,M) and ETS(A,Ad,M), can divide by values close to zero and are numerically unstable, so, as in the textbook (FPP-py §8.6) and the Week 12 lab, there are fifteen candidates. Even with the same AICc rule, different software (such as the textbook's statsforecast) can choose a different model. This notebook fits the data divided by their average size and converts back, so data that differ only in units (thousands or millions) get the same model. So that the optimizer does not stop short, each candidate is fitted from two starting points and restarted from where it stopped, and the converged fit with the highest training log-likelihood is kept (the same tool as the Week 12 lab).*

**트랙 2.** AR 은 소비의 과거만, VAR 는 소비와 소득의 과거를 씁니다. 도우미가 첫 학습 구간에서 시차 1 ~ 8 의 BIC 로 제안합니다(10주차처럼 다변량에서는 BIC).
*Track 2. The AR uses past consumption only; the VAR uses past consumption and income. The helper suggests lags 1 to 8 by BIC on the first training window (BIC, as in Week 10 for the multivariate case).*

**주 지표와 설정 확정 / Primary measure and fixing the settings.** 주 지표(RMSE · MAE · MASE)도 여기서 정합니다. 설정과 주 지표가 정해지면 「SETTINGS_FIXED」 를 체크하고 모두 실행하세요. 체크 전에는 백테스트가 돌지 않습니다.
*Choose the primary measure (RMSE, MAE or MASE) here too. When your settings and measure are decided, tick SETTINGS_FIXED and Run all; before that, the backtest does not run.*

In [ ]:
#@title 내 선택 2 / Your choice 2 { display-mode: "form" }
# 트랙 1 · SARIMA = ARIMA(p,d,q)(P,D,Q)m, m = 12 (월) · 4 (분기) / m = 12 for months, 4 for quarters
DIFF_d = 1  #@param {type:"slider", min:0, max:2, step:1}
DIFF_D = 1  #@param {type:"slider", min:0, max:1, step:1}
USE_SUGGESTED_SARIMA = True  #@param {type:"boolean"}
AR_p = 1  #@param {type:"slider", min:0, max:3, step:1}
MA_q = 1  #@param {type:"slider", min:0, max:3, step:1}
SAR_P = 0  #@param {type:"slider", min:0, max:1, step:1}
SMA_Q = 1  #@param {type:"slider", min:0, max:1, step:1}
# 트랙 1 · ETS
USE_SUGGESTED_ETS = True  #@param {type:"boolean"}
ETS_MODEL = "ETS(A,Ad,A)"  #@param ["ETS(A,N,N)", "ETS(A,A,N)", "ETS(A,Ad,N)", "ETS(A,N,A)", "ETS(A,A,A)", "ETS(A,Ad,A)", "ETS(M,N,N)", "ETS(M,A,N)", "ETS(M,Ad,N)", "ETS(M,N,A)", "ETS(M,A,A)", "ETS(M,Ad,A)", "ETS(M,N,M)", "ETS(M,A,M)", "ETS(M,Ad,M)"]
# 트랙 2
USE_SUGGESTED_LAGS = True  #@param {type:"boolean"}
AR_LAGS = 2  #@param {type:"slider", min:1, max:8, step:1}
VAR_LAGS = 2  #@param {type:"slider", min:1, max:8, step:1}
# 공통 / both tracks
PRIMARY_MEASURE = "RMSE"  #@param ["RMSE", "MAE", "MASE"]
SETTINGS_FIXED = False  #@param {type:"boolean"}
print("설정을 받았습니다. 아래 도우미가 제안값과 최종 설정을 보여 줍니다 / Settings received; the helper below shows the suggestions and the final settings.")
print("설정 확정 / settings fixed:", SETTINGS_FIXED, "" if SETTINGS_FIXED else "→ 백테스트는 체크한 뒤에 돕니다 / the backtest runs after you tick it")

In [ ]:
#@title 차분 확인 / Differencing check (first training window) { display-mode: "form" }
# 7주차 실습 1 · 2절과 과제 2 Q1 의 그림입니다. 왼쪽은 계열, 오른쪽은 ACF 와 6주차의 띠 ±2/√n. 첫 학습 구간만 씁니다.
# The plots of Week 7 lab sections 1 and 2 and Assignment 2, Q1: the series on the left, its ACF with Week 6's band ±2/√n on the right.
# Only the first training window is used.
if not T1:
    print("트랙 2 는 이미 변화율(%)이라 차분하지 않습니다. 이 셀은 건너뜁니다 / Track 2 already uses % changes, so there is no differencing; this cell is skipped.")
else:
    first = Y.iloc[:first_train].astype(float)                        # 첫 학습 구간 / the first training window
    z = first.copy()
    for _ in range(DIFF_D):
        z = z - z.shift(M)                                            # 계절 차분: 작년 같은 달(분기)을 뺀다 / seasonal difference: subtract the same month (quarter) a year earlier
    for _ in range(DIFF_d):
        z = z - z.shift(1)                                            # 보통 차분: 바로 앞 값을 뺀다 / ordinary difference: subtract the previous value
    z = z.dropna()                                                    # 앞에서 빈 값 D·M + d 개를 뺀다 / drop the D·M + d empty values at the start
    nl = 3 * M                                                        # 시차 3년치: 월별 36 · 분기 12 / three years of lags: 36 months or 12 quarters
    label = "no differencing" if DIFF_d == DIFF_D == 0 else f"after D = {DIFF_D}, d = {DIFF_d}"
    fig, axs = plt.subplots(2, 2, figsize=(12, 5.6), gridspec_kw={"width_ratios": [1.15, 1]})
    seasonal_lags = [k * M for k in (1, 2, 3)]                        # 계절 시차 / the seasonal lags
    rows = []
    for row, (s, name) in enumerate(((first, "original series"), (z, label))):
        k_max = min(nl, len(s) - 1)                                   # 값이 n 개면 시차 n − 1 까지만 계산된다 / n values give lags up to n − 1 only
        r = acf(s, nlags=k_max, fft=False)[1:]                        # 시차 1 ~ k_max 의 표본 ACF / sample ACF at lags 1 to k_max
        band = 2 / np.sqrt(len(s))                                    # 6주차의 띠 / Week 6's band
        axs[row, 0].plot(s.index, s.to_numpy(), color=GREEN, lw=1.1)
        if row == 1: axs[row, 0].axhline(0, color=GRAY, lw=0.8)
        axs[row, 0].set_title(f"{name}, first training window ({len(s)} values)", fontsize=9)
        bars(axs[row, 1], r, band, f"ACF, {name}, lags 1 to {len(r)}", f"lag ({unit}s)")
        at = lambda k: round(float(r[k - 1]), 2) + 0.0 if k <= len(r) else "n/a"   # + 0.0: −0.00 을 0.00 으로. 계산할 수 없는 시차는 n/a / −0.00 as 0.00; n/a where the lag cannot be computed
        rows.append({"series": name, "values": len(s), "band ±": round(band, 3), "lag 1": at(1), **{f"lag {k}": at(k) for k in seasonal_lags}})
    plt.tight_layout(); plt.show()
    print("ACF 의 몇 시차 / the ACF at a few lags"); print(pd.DataFrame(rows).to_string(index=False))
    n_short = min(rw["values"] for rw in rows)
    if n_short - 1 < nl:                                              # 짧은 자료: 오류가 아니라 계산할 수 없는 시차 / short data: lags that cannot be computed, not an error
        print(f"n/a = 계산 불가: 값이 {n_short} 개라 시차 {n_short - 1} 까지만 계산합니다. 자료가 짧아서이지 오류가 아닙니다. 계산된 시차로 읽으세요.")
        print(f"n/a = not computable: with {n_short} values the ACF goes only up to lag {n_short - 1}. The data are short; this is not an error. Read the lags that are computed.")
    print()
    print("읽는 법 / How to read (7주차 / Week 7)")
    print("- 막대가 천천히 줄어들면 수준이 아직 떠돕니다: 보통 차분 d. 계절 시차에서 다시 솟으면 해마다 도는 모양이 남았습니다: 계절 차분 D.")
    print("  Bars that fade slowly mean the level still wanders: an ordinary difference d. Bars that rise again at the seasonal lags mean the yearly shape is still there: a seasonal difference D.")
    print("- 차분한 계열이 0 둘레의 한 띠에서 움직이고 ACF 가 빨리 줄면 멈춥니다. 띠 밖 막대가 몇 개 남는 것은 괜찮습니다. 그것은 다음 단계의 p · q · P · Q 가 맡습니다.")
    print("  Stop when the differenced series moves within one band around 0 and its ACF dies out quickly. A few bars outside the band are fine; the orders p, q, P, Q take them on next.")
    r_last = rows[-1]["lag 1"]
    if (DIFF_d + DIFF_D) > 0 and isinstance(r_last, float) and r_last <= -0.4:
        print(f"- 주의 / note: 시차 1 이 {r_last} 입니다. −0.5 근처의 막대는 차분을 한 번 더 했을 수 있다는 경고입니다(증명은 아님, 7주차 「What If We Difference One Time Too Many?」).")
        print(f"  The lag-1 value is {r_last}. A bar near −0.5 warns that one difference may be too many (a warning, not proof; Week 7).")
# 기대값 / Expected   (기본값: 가구, D = 1, d = 1 / the default: furniture, D = 1, d = 1)
#               series  values  band ±  lag 1  lag 12  lag 24  lag 36
#      original series     177   0.150   0.99    0.87    0.61    0.33
#   after D = 1, d = 1     164   0.156   0.29   -0.37    0.00   -0.04
#   원계열의 막대는 천천히 줄고, 두 차분 뒤에는 시차 1 · 12 에 막대가 남지만 ACF 가 빨리 줄어듭니다.
#   The original series' bars fade slowly; after both differences bars remain at lags 1 and 12, but the ACF dies out quickly.

In [ ]:
#@title 도우미와 최종 설정 / Helper and final settings { display-mode: "form" }
train0 = (Y.iloc[:first_train] if T1 else Y.iloc[:first_train]).to_numpy(dtype=float)
if T1:
    rows = []
    for p, q, P, Q in itertools.product(range(3), range(3), range(2), range(2)):
        try:
            r = fit_sarima(train0, (p, DIFF_d, q), (P, DIFF_D, Q, M))
            rows.append({"model": sarima_name((p, DIFF_d, q), (P, DIFF_D, Q, M)), "p": p, "q": q, "P": P, "Q": Q, "AIC": r.aic, "ok": r.converged_ok})
        except Exception:
            rows.append({"model": sarima_name((p, DIFF_d, q), (P, DIFF_D, Q, M)), "p": p, "q": q, "P": P, "Q": Q, "AIC": np.nan, "ok": False})
    grid_all = pd.DataFrame(rows); grid = grid_all[grid_all["ok"]].drop(columns="ok").sort_values("AIC").reset_index(drop=True)
    print(f"수렴하지 않아 뺀 SARIMA 후보 / SARIMA candidates dropped for not converging: {int((~grid_all['ok']).sum())} of {len(grid_all)}")
    print(f"SARIMA = ARIMA(p,d,q)(P,D,Q){M}, d = {DIFF_d}, D = {DIFF_D}: AIC 가 낮은 다섯 / the five lowest AIC (first training window only)")
    print(grid.head(5).round(2).to_string(index=False)); print()
    rows = []
    for name in ETS_CANDIDATES:
        try:
            r = fit_ets(train0, name, M); rows.append({"model": name, "AICc": r.aicc, "ok": r.converged_ok and np.isfinite(r.aicc)})
        except Exception:
            rows.append({"model": name, "AICc": np.nan, "ok": False})
    egrid_all = pd.DataFrame(rows); egrid = egrid_all[egrid_all["ok"]].drop(columns="ok").sort_values("AICc").reset_index(drop=True)
    print(f"수렴하지 않았거나 맞출 수 없어 뺀 ETS 후보 / ETS candidates dropped (did not converge or could not be fitted): {int((~egrid_all['ok']).sum())} of {len(egrid_all)}")
    print("ETS: AICc 가 낮은 다섯 / the five lowest AICc (first training window only)")
    print(egrid.head(5).round(2).to_string(index=False)); print()
    b = grid.iloc[0]; e0 = egrid.iloc[0]
    SARIMA_ORDER = (int(b.p), DIFF_d, int(b.q)) if USE_SUGGESTED_SARIMA else (AR_p, DIFF_d, MA_q)
    SARIMA_SEASONAL = (int(b.P), DIFF_D, int(b.Q), M) if USE_SUGGESTED_SARIMA else (SAR_P, DIFF_D, SMA_Q, M)
    ETS_SPEC = e0.model if USE_SUGGESTED_ETS else ETS_MODEL
    FINAL = {"SARIMA": {"name": sarima_name(SARIMA_ORDER, SARIMA_SEASONAL), "order": SARIMA_ORDER, "seasonal_order": SARIMA_SEASONAL, "suggested": USE_SUGGESTED_SARIMA},
             "ETS": {"name": ETS_SPEC, "suggested": USE_SUGGESTED_ETS}}
    MODELS = ("SARIMA", "ETS")
    print(f"최종 / final : {FINAL['SARIMA']['name']} ({'suggested' if USE_SUGGESTED_SARIMA else 'my choice'}) "
          f"| {ETS_SPEC} ({'suggested' if USE_SUGGESTED_ETS else 'my choice'})")
    print("SARIMA 의 AIC 와 ETS 의 AICc 는 서로 견줄 수 없습니다. 같은 가족 안에서만 고르는 데 씁니다 / the two tables cannot be compared with each other.")
else:
    ar_fits = {p: fit_ar(train0[:, 0], p) for p in range(1, 9)}
    ar_bic = pd.DataFrame([{"lags": p, "BIC": r.bic, "converged": r.converged_ok} for p, r in ar_fits.items()])
    sel = VAR(train0).select_order(8)                                  # 시차 0 ~ 8 을 같은 표본에서 / lags 0 to 8 on a common sample
    var_tab = pd.DataFrame({"lags": range(1, 9), "BIC": np.asarray(sel.ics["bic"], float)[1:9]})
    var_bic = int(var_tab.sort_values("BIC").iloc[0]["lags"])           # 1 ~ 8 가운데 최선 / the best among 1 to 8
    print("AR(p), 소비만 / consumption only: BIC (first training window only)"); print(ar_bic.round(2).to_string(index=False)); print()
    print("VAR(p), 소비와 소득 / consumption and income: BIC for lags 1 to 8 (first training window only)"); print(var_tab.round(3).to_string(index=False))
    if int(sel.selected_orders["bic"]) == 0:
        print("참고: 시차 0 까지 넣으면 BIC 는 0 을 고릅니다. 이 비교는 VAR 를 쓰므로 1 ~ 8 가운데 가장 낮은 값을 제안합니다 / note: with lag 0 allowed, BIC picks 0; the suggestion is the lowest among 1 to 8.")
    print()
    ok_ar = ar_bic[ar_bic["converged"]]
    AR_FINAL = int(ok_ar.sort_values("BIC").iloc[0]["lags"]) if USE_SUGGESTED_LAGS else AR_LAGS
    VAR_FINAL = var_bic if USE_SUGGESTED_LAGS else VAR_LAGS
    FINAL = {"AR": {"lags": AR_FINAL}, "VAR": {"lags": VAR_FINAL}, "suggested": USE_SUGGESTED_LAGS}
    MODELS = ("AR", "VAR")
    print(f"최종 / final : AR({AR_FINAL}) on consumption | VAR({VAR_FINAL}) on consumption and income ({'suggested by BIC' if USE_SUGGESTED_LAGS else 'my choice'})")
# 기대값 / Expected   (기본값: 가구, d = 1 · D = 1, 제안값 / the default: furniture, d = 1, D = 1, suggestions)
#   최종 / final : ARIMA(1,1,1)(0,1,1)12 (suggested) | ETS(M,Ad,A) (suggested)
#   AIC · AICc 의 끝자리와 수렴하지 못한 후보의 수는 statsmodels 판에 따라 다를 수 있습니다.
#   The last digits of the AIC and AICc, and the number of candidates that fail to converge, can differ with the statsmodels version.

In [ ]:
#@title 잔차 진단 / Residual diagnostics (first training window) { display-mode: "form" }
if T1:
    r1 = fit_sarima(train0, FINAL["SARIMA"]["order"], FINAL["SARIMA"]["seasonal_order"])
    r2 = fit_ets(train0, FINAL["ETS"]["name"], M)
    skip = DIFF_d + M * DIFF_D
    res = {"SARIMA": np.asarray(r1.resid)[skip:], "ETS": np.asarray(r2.resid)[M:]}
else:
    r1 = fit_ar(train0[:, 0], FINAL["AR"]["lags"]); r2 = fit_var(train0, FINAL["VAR"]["lags"])
    res = {"AR": np.asarray(r1.resid)[FINAL["AR"]["lags"]:], "VAR": np.asarray(r2.resid)[:, 0]}
# 자유도: SARIMA 는 p + q + P + Q, AR 은 p 를 빼고(statsmodels 의 model_df). ETS 는 표준 자유도가 없어 보정 없는 참고값.
# VAR 는 단변량 공식 대신 다변량 백색성 검정(포트맨토, 자유도 보정)을 쓴다.
# Degrees of freedom: subtract p + q + P + Q for SARIMA and p for AR (statsmodels' model_df). ETS has no standard df, so its value is
# an unadjusted reference. For the VAR, use the multivariate whiteness (portmanteau) test with the df adjustment instead.
if T1:
    so, ss = FINAL["SARIMA"]["order"], FINAL["SARIMA"]["seasonal_order"]
    mdf = {"SARIMA": so[0] + so[2] + ss[0] + ss[2], "ETS": 0}
else:
    mdf = {"AR": FINAL["AR"]["lags"], "VAR": None}
lb, how = {}, {}
for k, v in res.items():
    if k == "VAR":
        L = max(D["lb_lag"], FINAL["VAR"]["lags"] + 4)
        lb[k] = float(r2.test_whiteness(nlags=L, adjusted=True).pvalue); how[k] = f"multivariate portmanteau, lags {L}, df adjusted"
    else:
        L = max(D["lb_lag"], mdf[k] + 4)
        lb[k] = float(acorr_ljungbox(v, lags=[L], model_df=mdf[k], return_df=True)["lb_pvalue"].iloc[0])
        how[k] = f"Ljung–Box, lags {L}, model_df {mdf[k]}" if k != "ETS" else f"Ljung–Box, lags {L}, no df adjustment (reference value)"
for k, r in zip(res, (r1, r2)):
    if not getattr(r, "converged_ok", True): print(f"주의 / note: {k} 적합이 반복을 늘려도 수렴하지 않았습니다 / did not converge even after a retry")
for k, pv in lb.items():
    if k == "ETS":                                                    # 자유도를 빼지 않은 참고값: 판정문을 붙이지 않는다 / unadjusted reference value: no verdict
        print(f"{k:6s}: p = {pv:.3f}  ({how[k]})   참고값: 자유도를 빼지 않아, 뺐다면 더 작아졌을 값입니다. 잔차 ACF 그림과 함께 읽으세요 / "
              "a reference value: no degrees of freedom are subtracted, so an adjusted p would be smaller. Read it with the residual ACF.")
    else:
        print(f"{k:6s}: p = {pv:.3f}  ({how[k]})   " + ("작은 p: 잔차에 남은 패턴이 있을 수 있음 / small p: some pattern may be left" if pv < 0.05 else "남은 패턴의 뚜렷한 증거 없음 / no clear evidence of a pattern left"))
fig, axs = plt.subplots(1, 2, figsize=(9, 2.6))
for ax, (k, v) in zip(axs, res.items()):
    rv = acf(v, nlags=D["lb_lag"], fft=False)[1:]                     # 잔차의 표본 ACF / the residuals' sample ACF
    bars(ax, rv, 2 / np.sqrt(len(v)), f"{k} residual ACF (first training window)", f"lag ({unit}s)")
plt.tight_layout(); plt.show()
print("띠는 ±2/√n 으로 시차마다 따로 긋는 기준이고, Ljung–Box 는 여러 시차를 한꺼번에 봅니다(과제 2 Q3) / "
      "the band ±2/√n is drawn for each lag separately, while Ljung–Box looks at many lags together (Assignment 2, Q3).")

In [ ]:
#@title 백테스트 / Backtest. rolling origin: 원점마다 다시 맞추고 예측 / refit and forecast at every origin { display-mode: "form" }
RESULTS, EXPLORATORY, FIRST_FIXED = None, False, None
if not SETTINGS_FIXED:
    print("아직 설정을 확정하지 않았습니다. 「내 선택 2」 에서 SETTINGS_FIXED 를 체크하고 모두 실행하세요 / Settings are not fixed yet: tick SETTINGS_FIXED in Your choice 2 and Run all.")
else:
    STORE = "project_first_fixed.json"                                  # 트랙 · 자료마다 한 칸 / one entry per track and dataset
    store = json.load(open(STORE, encoding="utf-8")) if os.path.exists(STORE) else {}
    KEY = f"{TRACK} | {DATA_LABEL} | {DATA_SHA}"
    now = {"final_settings": json.loads(json.dumps(FINAL, default=str)), "primary_measure": PRIMARY_MEASURE}
    FIRST_FIXED = store.get(KEY)
    if FIRST_FIXED is not None:
        EXPLORATORY = json.dumps(FIRST_FIXED["settings"], sort_keys=True) != json.dumps(now, sort_keys=True)
    if EXPLORATORY:
        print("탐색적 비교 / EXPLORATORY: 처음 확정한 설정과 다릅니다. 보고서에 처음 설정의 결과와 함께 밝히세요 / these settings differ from your first fixed ones; report them as exploratory.")
    t0, rows, notconv = time.time(), [], {}
    arr = Y.to_numpy(dtype=float)
    for o in ORIGINS:
        train = arr[:o]
        if T1:
            actual, scale = arr[o:o + D["h"]], mase_scale(train, M)
            ra = fit_sarima(train, FINAL["SARIMA"]["order"], FINAL["SARIMA"]["seasonal_order"])
            rb = fit_ets(train, FINAL["ETS"]["name"], M)
            fcs = {"SARIMA": fc_sarima(ra, D["h"]), "ETS": fc_ets(rb, D["h"], len(train)), "seasonal naive (baseline)": fc_snaive(train, D["h"], M)}
            conv = {"SARIMA": ra.converged_ok, "ETS": rb.converged_ok}
        else:
            actual, scale = arr[o:o + D["h"], 0], mase_scale(train[:, 0], 1)
            ra = fit_ar(train[:, 0], FINAL["AR"]["lags"])
            fcs = {"AR": fc_sarima(ra, D["h"]), "VAR": fc_var(fit_var(train, FINAL["VAR"]["lags"]), train, D["h"]),
                   "training mean (baseline)": {"mean": np.repeat(train[:, 0].mean(), D["h"])}}
            conv = {"AR": ra.converged_ok, "VAR": True}
        for k, v in conv.items(): notconv[k] = notconv.get(k, 0) + (not v)
        for name, f in fcs.items():
            for k in range(D["h"]):
                r = {"origin": dates[o - 1].date(), "target_date": dates[o + k].date(), "horizon": k + 1, "model": name,
                     "actual": actual[k], "forecast": f["mean"][k], "scale": scale, "converged": conv.get(name, True)}
                for tag in ("80", "95"):
                    r["lo" + tag], r["hi" + tag] = (f["lo" + tag][k], f["hi" + tag][k]) if "lo" + tag in f else (np.nan, np.nan)
                rows.append(r)
    BT = pd.DataFrame(rows)
    BT.to_csv("project_backtest.csv", index=False)
    RESULTS = summarize(BT)
    NOT_CONVERGED = notconv
    if FIRST_FIXED is None:                                             # 이 트랙 · 자료의 첫 확정: 설정과 결과를 함께 보관 / first fix for this track and data: keep settings and results
        first_csv = "project_backtest_first_" + hashlib.sha256(KEY.encode()).hexdigest()[:8] + ".csv"
        BT.to_csv(first_csv, index=False)
        FIRST_FIXED = {"settings": now, "fixed_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
                       "results": json.loads(RESULTS.round(6).to_json(orient="split")),
                       "results_sha256": hashlib.sha256(RESULTS.round(6).to_csv().encode()).hexdigest()[:16],
                       "backtest_file": first_csv, "fits_not_converged": dict(notconv)}
        store[KEY] = FIRST_FIXED
        json.dump(store, open(STORE, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    FIRST_RESULTS = pd.read_json(io.StringIO(json.dumps(FIRST_FIXED["results"])), orient="split")
    print(f"완료 / done in {time.time() - t0:.0f} s: {len(ORIGINS)} origins × {D['h']} steps = {len(ORIGINS) * D['h']} forecasts per model. "
          "모든 예측은 project_backtest.csv 에도 저장 / every forecast is also saved to project_backtest.csv")
    for k, v in notconv.items():
        if v: print(f"주의 / note: {k} 적합 {v}개가 반복을 늘려도 수렴하지 않았습니다. 그 원점의 예측은 남기고 converged = False 로 표시했습니다 / "
                    f"{v} {k} fits did not converge after a retry; their forecasts are kept and marked converged = False.")

In [ ]:
#@title 요약 / Summary { display-mode: "form" }
if RESULTS is None:
    print("설정을 확정한 뒤에 보입니다. 「내 선택 2」 에서 SETTINGS_FIXED 를 체크하세요 / shown after you fix the settings (tick SETTINGS_FIXED)")
else:
    order = [PRIMARY_MEASURE] + [c for c in RESULTS.columns if c != PRIMARY_MEASURE]
    if EXPLORATORY:
        first_measure = FIRST_FIXED["settings"]["primary_measure"]
        fo = [first_measure] + [c for c in FIRST_RESULTS.columns if c != first_measure]
        print(f"MAIN RESULT. 처음 확정한 설정의 결과({FIRST_FIXED['fixed_at']}), 주 지표 = {first_measure} / the results of your first fixed settings")
        print(FIRST_RESULTS[fo].round(3).to_string()); print(f"처음 백테스트 파일 / first backtest file: {FIRST_FIXED['backtest_file']}"); print()
        print("EXPLORATORY. 지금 설정의 결과 / the results of the current settings")
    print(f"SUMMARY. 평가 구간 전체, 주 지표 = {PRIMARY_MEASURE} / the whole evaluation window, primary measure {PRIMARY_MEASURE}")
    print(RESULTS[order].round(3).to_string()); print()
    print("읽는 법 / How to read: coverage 는 실제 값이 구간 안에 든 비율(명목 80 · 95%), width 는 구간의 평균 폭. 기준선(baseline)은 구간이 없습니다.")
    print("coverage is the share of actual values inside the interval (nominal 80 and 95%); width is its mean width. The baseline has no interval."); print()
    e = BT.assign(err=BT["actual"] - BT["forecast"])
    e = e.assign(sq=e["err"] ** 2, ab=e["err"].abs(), sc=e["err"].abs() / e["scale"])
    col = {"RMSE": "sq", "MAE": "ab", "MASE": "sc"}[PRIMARY_MEASURE]
    byh = e.groupby(["horizon", "model"], sort=False)[col].mean().unstack()
    if PRIMARY_MEASURE == "RMSE": byh = np.sqrt(byh)
    print(f"BY HORIZON. 몇 걸음 앞인지에 따른 {PRIMARY_MEASURE} / {PRIMARY_MEASURE} by how many steps ahead"); print(byh.round(3).to_string()); print()
    n_pairs = len(ORIGINS) * D["h"]
    print(f"적중률은 {n_pairs}개 예측에서 셉니다. 같은 원점의 예측끼리는 서로 독립이 아니라, 이 수는 확인의 크기를 가늠하는 데만 쓰세요.")
    print(f"Coverage counts {n_pairs} forecasts; forecasts from the same origin are not independent, so treat this count as a rough sense of size only.")

In [ ]:
#@title 그림 / Figures { display-mode: "form" }
if RESULTS is None:
    print("설정을 확정한 뒤에 보입니다. 「내 선택 2」 에서 SETTINGS_FIXED 를 체크하세요 / shown after you fix the settings (tick SETTINGS_FIXED)")
else:
    two = list(MODELS); cols = {two[0]: GREEN, two[1]: BLUE}
    tgt = Y if T1 else Y.iloc[:, 0]
    fig, ax = plt.subplots(figsize=(9, 2.6))
    ax.plot(dates, tgt, color=GREEN, lw=1.1); ax.axvspan(dates[first_train], dates[-1], color=GOLD, alpha=0.15, label="evaluation window")
    ax.set(title=DATA_LABEL + ": the whole series", ylabel=UNITS); ax.legend(fontsize=8, frameon=False); plt.tight_layout(); plt.show()
    pick = [ORIGINS[0], ORIGINS[len(ORIGINS) // 2], ORIGINS[-1]]
    tgt = Y if T1 else Y.iloc[:, 0]
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.0), sharey=True)
    for ax, o in zip(axs, pick):
        lo_i = max(0, o - 3 * max(M, 4))
        ax.plot(dates[lo_i:o + D["h"]], tgt.iloc[lo_i:o + D["h"]], color="black", lw=1, label="actual")
        part = BT[BT["origin"] == dates[o - 1].date()]
        for name in two:
            p = part[part["model"] == name]; td = pd.to_datetime(p["target_date"])
            ax.fill_between(td, p["lo80"], p["hi80"], color=cols[name], alpha=0.18)
            ax.plot(td, p["forecast"], color=cols[name], lw=1.6, label=name)
        ax.axvline(dates[o - 1], color=GRAY, ls=":", lw=1); ax.set_title(f"origin {dates[o - 1].date()} (bands: 80%)", fontsize=9)
        loc = mdates.AutoDateLocator(maxticks=5); ax.xaxis.set_major_locator(loc); ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    axs[0].legend(fontsize=8, frameon=False); plt.tight_layout(); plt.show()
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.0))
    for name in byh.columns:
        axs[0].plot(byh.index, byh[name], marker="o", lw=1.6, color=cols.get(name, GRAY), label=name)
    axs[0].set(xlabel="steps ahead", title=f"{PRIMARY_MEASURE} by horizon"); axs[0].legend(fontsize=8, frameon=False)
    x = np.arange(2); w = 0.35
    for i, name in enumerate(two):
        axs[1].bar(x + (i - 0.5) * w, [RESULTS.loc[name, "coverage80_%"], RESULTS.loc[name, "coverage95_%"]], w, color=cols[name], label=name)
    axs[1].scatter(x, [80, 95], marker="_", s=900, color="black", label="nominal")
    axs[1].set(xticks=x, xticklabels=["80% interval", "95% interval"], ylim=(0, 105), ylabel="coverage (%)", title="Did the intervals keep their promise?")
    axs[1].legend(fontsize=8, frameon=False, loc="upper center", ncol=3, bbox_to_anchor=(0.5, -0.12)); plt.tight_layout(); plt.show()

In [ ]:
#@title 다음 기간 예측 / Forecast beyond the data { display-mode: "form" }
if RESULTS is None:
    print("설정을 확정한 뒤에 보입니다. 「내 선택 2」 에서 SETTINGS_FIXED 를 체크하세요 / shown after you fix the settings (tick SETTINGS_FIXED)")
else:
    full = Y.to_numpy(dtype=float)
    step = pd.DateOffset(months=1) if FREQ == "monthly" else pd.DateOffset(months=3)
    future = [dates[-1] + step * (k + 1) for k in range(D["h"])]
    if T1:
        fx = {"SARIMA": fc_sarima(fit_sarima(full, FINAL["SARIMA"]["order"], FINAL["SARIMA"]["seasonal_order"]), D["h"]),
              "ETS": fc_ets(fit_ets(full, FINAL["ETS"]["name"], M), D["h"], len(full))}
    else:
        fx = {"AR": fc_sarima(fit_ar(full[:, 0], FINAL["AR"]["lags"]), D["h"]), "VAR": fc_var(fit_var(full, FINAL["VAR"]["lags"]), full, D["h"])}
    FUTURE = pd.concat([pd.DataFrame({"date": [f.date() for f in future], "model": k, "forecast": v["mean"], "lo80": v["lo80"], "hi80": v["hi80"],
                                      "lo95": v["lo95"], "hi95": v["hi95"]}) for k, v in fx.items()])
    print("모든 자료로 다시 맞춘 다음 기간 예측 / forecasts beyond the data, refit on all of it"); print(FUTURE.round(3).to_string(index=False))
    fig, ax = plt.subplots(figsize=(9, 2.8)); tail = 4 * max(M, 4)
    ax.plot(dates[-tail:], tgt.iloc[-tail:], color="black", lw=1, label="data")
    for k, v in fx.items():
        ax.fill_between(future, v["lo95"], v["hi95"], color=cols[k], alpha=0.12); ax.fill_between(future, v["lo80"], v["hi80"], color=cols[k], alpha=0.22)
        ax.plot(future, v["mean"], color=cols[k], lw=1.6, label=f"{k} (bands: 80% and 95%)")
    ax.set(title="Next " + str(D["h"]) + (" months" if FREQ == "monthly" else " quarters"), ylabel=UNITS); ax.legend(fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()

In [ ]:
#@title 기록 / Record. 출력을 지우지 마세요 / keep this output { display-mode: "form" }
record = {"notebook": NOTEBOOK, "track": TRACK, "data": DATA_LABEL, "data_sha256": DATA_SHA, "primary_measure": PRIMARY_MEASURE,
          "settings_fixed": SETTINGS_FIXED, "exploratory": EXPLORATORY,
          "first_fixed": ({k: FIRST_FIXED[k] for k in ("settings", "fixed_at", "results_sha256", "backtest_file", "fits_not_converged")} if FIRST_FIXED else None),
          "first_results_table": (FIRST_FIXED["results"] if FIRST_FIXED else None),
          "fits_not_converged": NOT_CONVERGED if RESULTS is not None else None,
          "design": {"horizon": D["h"], "origins": len(ORIGINS), "step": D["step"], "first_training_points": first_train, "evaluation_points": N - first_train},
          "final_settings": FINAL, "results_sha256": hashlib.sha256(RESULTS.round(6).to_csv().encode()).hexdigest()[:16] if RESULTS is not None else None,
          "python": platform.python_version(), "statsmodels": statsmodels.__version__, "pandas": pd.__version__,
          "run_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}
print(json.dumps(record, ensure_ascii=False, indent=2, default=str))

## 보고서에 쓰는 법 / Using this in your report
- 요약 표, 걸음별 표, 그림 하나 이상을 보고서에 옮기세요. 숫자는 표에서 그대로 인용합니다. *Bring the summary table, the by-horizon table and at least one figure into the report; quote numbers from the tables.*
- 필수 문단: 가장 중요한 결론을 내 결과의 수치 두 개와 평가 구간 하나로 뒷받침하고, 결론이 맞지 않을 상황과 재검토에 필요한 증거를 쓰세요. *Required paragraph: support your main conclusion with two numbers from your results and one evaluation window, then say where it may not apply and what evidence would make you reconsider it.*
- 결과를 본 뒤 설정을 바꿨다면(기록 셀의 exploratory = true) 요약의 MAIN RESULT, 곧 기록 셀의 first_results_table 이 주 결과입니다. 바꾼 비교는 탐색적 비교라고 밝히세요. *If you changed settings after seeing results (exploratory = true), the MAIN RESULT in the summary, also first_results_table in the record, is your main result; call the later comparison exploratory.*
- 수렴하지 않은 적합이 있었다면(백테스트 안내와 기록의 fits_not_converged) 예측 성능과 함께 계산의 안정성도 해석하세요. 그 개수는 컴퓨터 환경에 따라 달라질 수 있습니다. *If some fits did not converge, interpret computational stability alongside forecast accuracy; the count can differ between computing environments.*
- 제안값을 썼다면 그것이 무엇이고 왜 받아들였는지, 직접 골랐다면 무엇을 근거로 했는지 쓰세요. 잔차 진단의 p 값도 함께 해석하세요. ETS 의 p 는 자유도를 빼지 않은 참고값이라 잔차 ACF 와 함께 읽습니다. *If you used the suggestions, say what they were and why you accepted them; if you chose yourself, say on what basis. Interpret the residual p-values too; the ETS p-value is an unadjusted reference value, so read it with the residual ACF.*
- 기록 셀의 출력은 지우지 마세요. 보고서와 함께 출력을 저장한 이 노트북(.ipynb 파일 또는 Colab 링크)을 냅니다. *Keep the record cell's output, and submit this notebook with outputs saved (.ipynb file or Colab link) with the report.*